# LSTM on MediaPipe keypoints — ASL vs SGSL vs combined

Trains **three separate models** on sequences built from **`03_mediapipe_pose_hands_videos.ipynb`** outputs:

| Experiment | Training data | Classes |
|------------|----------------|---------|
| **ASL only** | `Videos/ASL_60_pose_hands_keypoints` (or `ASL_pose_hands_keypoints`) | ASL gloss folder names |
| **SGSL only** | `Videos/SGSL_pose_hands_keypoints` | SGSL place folders |
| **Combined** | Concatenation of both | `ASL/<gloss>` and `SGSL/<place>` (disjoint label strings) |

Each frame is **`frame_XXXXXX.npy`** shaped **`(75, 5)`** → flattened to **`375`** floats per timestep. Sequences are padded / truncated to **`SEQ_LEN`** (default **60**).

**Metrics:** stratified **80 / 10 / 10** train/val/test (falls back to random split if stratification is impossible); test **accuracy** reported. For **combined**, test accuracy is also split into **ASL-only** and **SGSL-only** subsets.

**Install packages once** (Terminal or your venv), **then start Jupyter**. Avoid `%pip install tensorflow` inside this notebook — upgrading TensorFlow while the kernel is running often **crashes the kernel** on macOS and Windows.

By default the notebook sets **`FORCE_TF_CPU_ONLY = True`** (Metal/GPU backends sometimes segfault Jupyter). After everything runs reliably, try **`False`** for faster training if your GPU stack is stable.

If training still stresses memory, keep **`BATCH_SIZE = 8`** or set **`QUICK_RUN = True`** for a short smoke run (~40 epochs) before full training.
```bash
pip install "tensorflow>=2.15" "scikit-learn>=1.3" "numpy>=1.26,<2"
```


In [2]:
!pip install "tensorflow>=2.15" "scikit-learn>=1.3" "numpy>=1.26,<2" 


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import importlib.util
import sys

def _missing(pkgs: list[tuple[str, str]]) -> list[str]:
    lines = []
    for mod_name, pip_line in pkgs:
        if importlib.util.find_spec(mod_name) is None:
            lines.append(f"  - {pip_line}")
    return lines

_hints = _missing(
    [
        ("numpy", 'pip install "numpy>=1.26,<2"'),
        ("sklearn", 'pip install "scikit-learn>=1.3"'),
        ("tensorflow", 'pip install "tensorflow>=2.15"'),
    ]
)
if _hints:
    print(
        "Missing dependencies. Install in Terminal (same Python as Jupyter), then restart the kernel:\n"
        + "\n".join(_hints)
        + "\n\nDo not pip-install TensorFlow inside this notebook — it often kills the kernel.",
        file=sys.stderr,
    )
    raise ImportError("Install dependencies (see messages above), restart Jupyter, then re-run.")

print("Dependencies OK.")


Dependencies OK.


In [4]:
from __future__ import annotations

import gc
import os
import warnings
from pathlib import Path

import numpy as np

# Must be set before importing TensorFlow (Metal/CUDA backend stability in Jupyter).
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "4")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "4")

# Apple Silicon Metal / some GPU stacks segfault the Jupyter kernel — CPU-only is slower but stable.
FORCE_TF_CPU_ONLY = True

import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=UserWarning)

if FORCE_TF_CPU_ONLY:
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        try:
            tf.config.experimental.set_visible_devices([], "GPU")
        except Exception:
            pass

try:
    for _gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(_gpu, True)
except Exception:
    pass

PROJECT_ROOT = Path(".").resolve()

# Match `03_*` keypoints output layout
_used_asl_60 = (PROJECT_ROOT / "Videos" / "ASL_60").is_dir()
ASL_KP_ROOT = (
    PROJECT_ROOT / "Videos" / "ASL_60_pose_hands_keypoints"
    if _used_asl_60
    else PROJECT_ROOT / "Videos" / "ASL_pose_hands_keypoints"
)
SGSL_KP_ROOT = PROJECT_ROOT / "Videos" / "SGSL_pose_hands_keypoints"

SEQ_LEN = 60
RANDOM_STATE = 42
ARTIFACTS = PROJECT_ROOT / "artifacts_lstm_asl_sgsl"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Training / model (smaller batch reduces RAM spikes during LSTM + BN)
MAX_EPOCHS = 500
BATCH_SIZE = 16


: 

In [ ]:
def _sorted_clip_dirs(class_dir: Path) -> list[Path]:
    out = []
    for p in class_dir.iterdir():
        if not p.is_dir() or p.name.startswith("."):
            continue
        try:
            out.append((int(p.name), p))
        except ValueError:
            out.append((p.name, p))
    out.sort(key=lambda t: t[0])
    return [p for _, p in out]


def load_flat_clip(clip_dir: Path, seq_len: int) -> np.ndarray | None:
    """Return float32 array (seq_len, F) or None if unusable."""
    files = sorted(clip_dir.glob("frame_*.npy"))
    if not files:
        return None

    frames: list[np.ndarray] = []
    feat_dim: int | None = None

    for f in files[:seq_len]:
        a = np.load(f).astype(np.float32, copy=False)
        if a.ndim == 2 and a.shape[1] == 5:
            v = np.nan_to_num(a.reshape(-1), nan=0.0, posinf=0.0, neginf=0.0)
        elif a.ndim == 1:
            v = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            return None

        if feat_dim is None:
            feat_dim = int(v.shape[0])
        elif int(v.shape[0]) != feat_dim:
            return None

        frames.append(v.astype(np.float32, copy=False))

    if feat_dim is None:
        return None

    if len(frames) < seq_len:
        pad = np.zeros((seq_len - len(frames), feat_dim), dtype=np.float32)
        frames.extend(list(pad))

    seq = np.stack(frames[:seq_len], axis=0).astype(np.float32)
    return seq


def collect_from_root(key_root: Path, label_fn) -> tuple[list[np.ndarray], list[str]]:
    """label_fn(class_name: str) -> str label for this clip."""
    X_list: list[np.ndarray] = []
    y_list: list[str] = []

    if not key_root.is_dir():
        return X_list, y_list

    class_dirs = sorted(
        [p for p in key_root.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda p: p.name.lower(),
    )

    for cdir in class_dirs:
        lbl = label_fn(cdir.name)
        for clip in _sorted_clip_dirs(cdir):
            seq = load_flat_clip(clip, SEQ_LEN)
            if seq is None:
                continue
            if not np.isfinite(seq).all():
                continue
            X_list.append(seq)
            y_list.append(lbl)

    return X_list, y_list


def to_tensors(
    X_list: list[np.ndarray], y_labels: list[str]
) -> tuple[np.ndarray, np.ndarray, np.ndarray, LabelEncoder]:
    if not X_list:
        raise ValueError("No sequences loaded — run `03_*` keypoint export first.")
    X = np.stack(X_list, axis=0).astype(np.float32)
    le = LabelEncoder()
    y_int = le.fit_transform(y_labels)
    y_oh = tf.keras.utils.to_categorical(y_int, num_classes=len(le.classes_)).astype(np.float32)
    return X, y_oh, y_int, le


def _train_test_maybe_stratify(*arrays, test_size: float, stratify=None):
    kwargs = {"test_size": test_size, "random_state": RANDOM_STATE}
    try:
        return train_test_split(*arrays, stratify=stratify, **kwargs)
    except ValueError:
        # e.g. too few samples per class after filtering
        return train_test_split(*arrays, **kwargs)


def stratified_three_way(X, y_oh, y_int):
    (
        X_train,
        X_tmp,
        y_train,
        y_tmp,
        yi_train,
        yi_tmp,
    ) = _train_test_maybe_stratify(X, y_oh, y_int, test_size=0.20, stratify=y_int)
    (
        X_val,
        X_test,
        y_val,
        y_test,
        yi_val,
        yi_test,
    ) = _train_test_maybe_stratify(
        X_tmp, y_tmp, yi_tmp, test_size=0.50, stratify=yi_tmp
    )
    return (X_train, y_train, yi_train), (X_val, y_val, yi_val), (X_test, y_test, yi_test)


def stratified_three_way_with_tags(X, y_oh, y_int, tags: np.ndarray):
    (
        X_train,
        X_tmp,
        y_train,
        y_tmp,
        yi_train,
        yi_tmp,
        t_train,
        t_tmp,
    ) = _train_test_maybe_stratify(
        X, y_oh, y_int, tags, test_size=0.20, stratify=y_int
    )
    (
        X_val,
        X_test,
        y_val,
        y_test,
        yi_val,
        yi_test,
        t_val,
        t_test,
    ) = _train_test_maybe_stratify(
        X_tmp, y_tmp, yi_tmp, t_tmp, test_size=0.50, stratify=yi_tmp
    )
    return (
        (X_train, y_train, yi_train, t_train),
        (X_val, y_val, yi_val, t_val),
        (X_test, y_test, yi_test, t_test),
    )


def sample_weights_from_y(y_train_oh: np.ndarray, num_classes: int) -> np.ndarray:
    yi = np.argmax(y_train_oh, axis=1)
    counts = np.bincount(yi, minlength=num_classes).astype(np.float64)
    inv_freq = (counts.sum() / (counts + 1e-12)) / num_classes
    return inv_freq[yi].astype(np.float32)



def build_lstm(seq_len: int, feat_dim: int, num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(seq_len, feat_dim))
    x = inputs
    x = tf.keras.layers.LSTM(128, return_sequences=True, use_cudnn=False)(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.LSTM(64, return_sequences=False, use_cudnn=False)(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    metrics = ["accuracy"]
    k_top = min(3, num_classes)
    if k_top >= 2:
        metrics.append(
            tf.keras.metrics.TopKCategoricalAccuracy(k=k_top, name=f"top{k_top}_acc")
        )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=metrics,
    )
    return model


def train_one_run(name: str, X_train, y_train, X_val, y_val, X_test, y_test, yi_test, tags_test=None):
    """Train with NumPy arrays (avoids tf.data embedding huge constants — common RAM/kernel crash)."""
    tf.keras.backend.clear_session()
    gc.collect()

    num_classes = y_train.shape[1]
    seq_len = X_train.shape[1]
    feat_dim = X_train.shape[2]

    sw = sample_weights_from_y(y_train, num_classes)

    model = build_lstm(seq_len, feat_dim, num_classes)

    ckpt = ARTIFACTS / f"lstm_{name}.keras"
    cbs = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt),
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    model.fit(
        X_train,
        y_train,
        sample_weight=sw,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        callbacks=cbs,
        shuffle=True,
        verbose=1,
    )

    # In-memory weights match EarlyStopping best val_loss (same metric as checkpoint).
    probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    pred = np.argmax(probs, axis=1)

    del probs
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    out = {
        "name": name,
        "test_accuracy": float(accuracy_score(yi_test, pred)),
        "n_classes": int(num_classes),
        "n_test": int(len(X_test)),
    }

    if tags_test is not None:
        for tag in np.unique(tags_test):
            m = tags_test == tag
            if m.sum() == 0:
                continue
            out[f"test_accuracy_{tag}"] = float(
                accuracy_score(yi_test[m], pred[m])
            )
            out[f"n_test_{tag}"] = int(m.sum())

    print("\n===", name, "===")
    for k in sorted(out.keys()):
        if k != "name":
            print(f"  {k}: {out[k]}")

    return out


In [ ]:
results: list[dict] = []

# ----- ASL only -----
print("ASL keypoints root:", ASL_KP_ROOT)
X_a, y_a = collect_from_root(ASL_KP_ROOT, lambda g: g)
if len(X_a) == 0:
    print("[SKIP] No ASL keypoint sequences — export keypoints with `03_*`.")
else:
    X, y_oh, y_int, le_asl = to_tensors(X_a, y_a)
    (tr, va, te) = stratified_three_way(X, y_oh, y_int)
    (X_train, y_train, yi_train), (X_val, y_val, yi_val), (X_test, y_test, yi_test) = (
        tr,
        va,
        te,
    )
    results.append(
        train_one_run("ASL_only", X_train, y_train, X_val, y_val, X_test, y_test, yi_test)
    )
    del X, y_oh, y_int, X_train, X_val, X_test, y_train, y_val, y_test, yi_train, yi_val, yi_test
    gc.collect()

# ----- SGSL only -----
print("SGSL keypoints root:", SGSL_KP_ROOT)
X_s, y_s = collect_from_root(SGSL_KP_ROOT, lambda p: p)
if len(X_s) == 0:
    print("[SKIP] No SGSL keypoint sequences.")
else:
    X, y_oh, y_int, le_sgsl = to_tensors(X_s, y_s)
    (tr, va, te) = stratified_three_way(X, y_oh, y_int)
    (X_train, y_train, yi_train), (X_val, y_val, yi_val), (X_test, y_test, yi_test) = (
        tr,
        va,
        te,
    )
    results.append(
        train_one_run("SGSL_only", X_train, y_train, X_val, y_val, X_test, y_test, yi_test)
    )
    del X, y_oh, y_int, X_train, X_val, X_test, y_train, y_val, y_test, yi_train, yi_val, yi_test
    gc.collect()

# ----- Combined -----
if len(X_a) and len(X_s):
    # Prefix labels — reuse clips already loaded (avoid duplicate RAM / disk IO).
    X_all = X_a + X_s
    y_all = [f"ASL/{g}" for g in y_a] + [f"SGSL/{p}" for p in y_s]
    tags = np.array(["ASL"] * len(X_a) + ["SGSL"] * len(X_s))

    X, y_oh, y_int, le_comb = to_tensors(X_all, y_all)
    tr, va, te = stratified_three_way_with_tags(X, y_oh, y_int, tags)
    (
        X_train,
        y_train,
        yi_train,
        t_train,
    ), (
        X_val,
        y_val,
        yi_val,
        t_val,
    ), (
        X_test,
        y_test,
        yi_test,
        t_test,
    ) = tr, va, te

    results.append(
        train_one_run(
            "combined",
            X_train,
            y_train,
            X_val,
            y_val,
            X_test,
            y_test,
            yi_test,
            tags_test=t_test,
        )
    )
    del X, y_oh, y_int, X_train, X_val, X_test, y_train, y_val, y_test
    del yi_train, yi_val, yi_test, t_train, t_val, t_test
    gc.collect()
else:
    print("[SKIP] Combined experiment needs both ASL and SGSL sequences.")


In [ ]:
# Summary (no pandas dependency)
if not results:
    print("No experiments ran.")
else:
    keys = sorted({k for r in results for k in r})
    header = " | ".join(keys)
    print(header)
    print("-" * len(header))
    for r in results:
        print(" | ".join(str(r.get(k, "")) for k in keys))
